BHSig baseline CNN model cross validation on CEDAR dataset

In [1]:
import os
import re
import zipfile
import numpy as np
import pandas as pd
from pathlib import Path

import cv2
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [2]:
BHSIG_MODEL_PATH = r"W:\SRH study\Case Study 2\Offline Signature Verification\src\embeddings\BHSig260\baseline\bhsig_baseline_siamese.keras"

CEDAR_ORG_DIR  = r"W:\SRH study\Case Study 2\Offline Signature Verification\Datasets\signatures\full_org"
CEDAR_FORG_DIR = r"W:\SRH study\Case Study 2\Offline Signature Verification\Datasets\signatures\full_forg"

print("BHSig model exists:", os.path.exists(BHSIG_MODEL_PATH))
print("CEDAR genuine exists:", os.path.exists(CEDAR_ORG_DIR))
print("CEDAR forgery exists:", os.path.exists(CEDAR_FORG_DIR))

BHSig model exists: True
CEDAR genuine exists: True
CEDAR forgery exists: True


In [3]:
IMG_SIZE = (224, 224)
EMB_DIM = 128

class SquaredEuclidean(layers.Layer):
    def call(self, inputs):
        a, b = inputs
        return tf.reduce_sum(tf.square(a - b), axis=1, keepdims=True)

def build_embedding_network(input_shape=(224,224,1), emb_dim=128):
    inp = keras.Input(shape=input_shape)

    x = layers.Conv2D(32, 3, activation="relu")(inp)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64, 3, activation="relu")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128, 3, activation="relu")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(256, 3, activation="relu")(x)
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.2)(x)

    emb = layers.Dense(emb_dim)(x)
    emb = layers.Lambda(
        lambda t: tf.nn.l2_normalize(t, axis=1),
        output_shape=(emb_dim,),
        name="l2_norm"
    )(emb)

    return keras.Model(inp, emb, name="embedding_net")

def build_siamese_model(embedding_model, input_shape=(224,224,1)):
    a = keras.Input(shape=input_shape, name="img_a")
    b = keras.Input(shape=input_shape, name="img_b")

    emb_a = embedding_model(a)
    emb_b = embedding_model(b)

    dist = SquaredEuclidean(name="sq_euclidean")([emb_a, emb_b])
    return keras.Model([a, b], dist, name="siamese_network")

embedding_net = build_embedding_network(
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 1),
    emb_dim=EMB_DIM
)

siamese_model = build_siamese_model(
    embedding_net,
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 1)
)

print("Baseline architecture rebuilt.")


Baseline architecture rebuilt.


In [4]:
extract_dir = r"W:\SRH study\Case Study 2\Offline Signature Verification\src\embeddings\BHSig260\baseline\_extracted_cross_eval"
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(BHSIG_MODEL_PATH, "r") as z:
    names = z.namelist()
    print("Files inside .keras:")
    for n in names:
        print(" -", n)

    candidates = [n for n in names if n.endswith(".h5") and "weights" in n.lower()]
    if not candidates:
        raise RuntimeError("No weights file found inside the .keras archive.")

    weights_name = candidates[0]
    z.extract(weights_name, path=extract_dir)

weights_path = os.path.join(extract_dir, weights_name)
print("\nExtracted weights file:", weights_path)

Files inside .keras:
 - metadata.json
 - config.json
 - model.weights.h5

Extracted weights file: W:\SRH study\Case Study 2\Offline Signature Verification\src\embeddings\BHSig260\baseline\_extracted_cross_eval\model.weights.h5


In [ ]:
siamese_model.load_weights(weights_path)
print("BHSig baseline weights loaded successfully.")

✅ BHSig baseline weights loaded successfully.


In [6]:
PAT_ORG  = re.compile(r"^original_(\d+)_(\d+)\.png$", re.IGNORECASE)
PAT_FORG = re.compile(r"^forgeries_(\d+)_(\d+)\.png$", re.IGNORECASE)

def build_cedar_df(org_dir, forg_dir):
    rows = []

    for fname in os.listdir(org_dir):
        if not fname.lower().endswith(".png"):
            continue
        m = PAT_ORG.match(fname)
        if m:
            rows.append({
                "writer_id": int(m.group(1)),
                "sample_id": int(m.group(2)),
                "label": "genuine",
                "path": os.path.join(org_dir, fname)
            })

    for fname in os.listdir(forg_dir):
        if not fname.lower().endswith(".png"):
            continue
        m = PAT_FORG.match(fname)
        if m:
            rows.append({
                "writer_id": int(m.group(1)),
                "sample_id": int(m.group(2)),
                "label": "forgery",
                "path": os.path.join(forg_dir, fname)
            })

    return pd.DataFrame(rows)

valid_cedar = build_cedar_df(CEDAR_ORG_DIR, CEDAR_FORG_DIR)

print("Total images:", len(valid_cedar))
print(valid_cedar["label"].value_counts())
print("Unique writers:", valid_cedar["writer_id"].nunique())
display(valid_cedar.head())

Total images: 2640
label
genuine    1320
forgery    1320
Name: count, dtype: int64
Unique writers: 55


,writer_id,sample_id,label,path
0,10,1,genuine,W:\SRH study\Case Study 2\Offline Signature Ve...
1,10,10,genuine,W:\SRH study\Case Study 2\Offline Signature Ve...
2,10,11,genuine,W:\SRH study\Case Study 2\Offline Signature Ve...
3,10,12,genuine,W:\SRH study\Case Study 2\Offline Signature Ve...
4,10,13,genuine,W:\SRH study\Case Study 2\Offline Signature Ve...


In [7]:
def build_pools(df_subset):
    genuine_by_writer = {}
    forgery_by_writer = {}

    for wid, g in df_subset.groupby("writer_id"):
        gg = g[g["label"] == "genuine"]["path"].tolist()
        ff = g[g["label"] == "forgery"]["path"].tolist()

        if len(gg) >= 2:
            genuine_by_writer[wid] = gg
        if len(ff) >= 1:
            forgery_by_writer[wid] = ff

    return genuine_by_writer, forgery_by_writer

def generate_pairs(df, writer_set, n_pairs=20000, seed=42, neg_mix=0.9):
    df_subset = df[df["writer_id"].isin(writer_set)].copy()
    genuine_by_writer, forgery_by_writer = build_pools(df_subset)

    writers = sorted(genuine_by_writer.keys())
    writers_with_forg = sorted(set(genuine_by_writer) & set(forgery_by_writer))

    rng = np.random.default_rng(seed)
    n_pos = n_pairs // 2
    n_neg = n_pairs - n_pos
    n_neg_same  = int(round(n_neg * neg_mix))
    n_neg_cross = n_neg - n_neg_same

    pairs = []

    # Positive pairs: genuine-genuine same writer
    for _ in range(n_pos):
        w = rng.choice(writers)
        g = genuine_by_writer[w]
        i, j = rng.choice(len(g), size=2, replace=False)
        pairs.append({"path_a": g[i], "path_b": g[j], "label": 1})

    # Negative pairs: genuine-forgery same writer
    for _ in range(n_neg_same):
        w = rng.choice(writers_with_forg)
        g = genuine_by_writer[w]
        f = forgery_by_writer[w]
        i = rng.integers(0, len(g))
        j = rng.integers(0, len(f))
        pairs.append({"path_a": g[i], "path_b": f[j], "label": 0})

    # Negative pairs: genuine-genuine cross writer
    for _ in range(n_neg_cross):
        w1, w2 = rng.choice(writers, size=2, replace=False)
        g1 = genuine_by_writer[w1]
        g2 = genuine_by_writer[w2]
        i = rng.integers(0, len(g1))
        j = rng.integers(0, len(g2))
        pairs.append({"path_a": g1[i], "path_b": g2[j], "label": 0})

    return pd.DataFrame(pairs).sample(frac=1.0, random_state=seed).reset_index(drop=True)

cedar_writers = set(valid_cedar["writer_id"].unique())
cedar_test_pairs = generate_pairs(valid_cedar, cedar_writers, n_pairs=20000, seed=42, neg_mix=0.9)

print("CEDAR cross-dataset test pairs:", len(cedar_test_pairs))
print(cedar_test_pairs["label"].value_counts(normalize=True))
display(cedar_test_pairs.head())

CEDAR cross-dataset test pairs: 20000
label
0    0.5
1    0.5
Name: proportion, dtype: float64


,path_a,path_b,label
0,W:\SRH study\Case Study 2\Offline Signature Ve...,W:\SRH study\Case Study 2\Offline Signature Ve...,0
1,W:\SRH study\Case Study 2\Offline Signature Ve...,W:\SRH study\Case Study 2\Offline Signature Ve...,1
2,W:\SRH study\Case Study 2\Offline Signature Ve...,W:\SRH study\Case Study 2\Offline Signature Ve...,1
3,W:\SRH study\Case Study 2\Offline Signature Ve...,W:\SRH study\Case Study 2\Offline Signature Ve...,1
4,W:\SRH study\Case Study 2\Offline Signature Ve...,W:\SRH study\Case Study 2\Offline Signature Ve...,0


In [8]:
def load_preprocess(path):
    path = path.numpy().decode("utf-8")
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Failed to read: {path}")

    img = cv2.resize(img, IMG_SIZE, interpolation=cv2.INTER_AREA)
    img = img.astype(np.float32) / 255.0
    img = np.expand_dims(img, axis=-1)
    return img

def tf_load_preprocess(path):
    img = tf.py_function(load_preprocess, [path], Tout=tf.float32)
    img.set_shape([IMG_SIZE[0], IMG_SIZE[1], 1])
    return img

def make_pair_dataset(pairs_df, batch_size=32, shuffle=False):
    a = pairs_df["path_a"].astype(str).values
    b = pairs_df["path_b"].astype(str).values
    y = pairs_df["label"].astype(np.float32).values

    ds = tf.data.Dataset.from_tensor_slices((a, b, y))

    def map_fn(pa, pb, lab):
        ia = tf_load_preprocess(pa)
        ib = tf_load_preprocess(pb)
        return (ia, ib), lab

    ds = ds.map(map_fn, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

cedar_test_ds = make_pair_dataset(cedar_test_pairs, batch_size=32, shuffle=False)

(xb, yb) = next(iter(cedar_test_ds))
print(xb[0].shape, xb[1].shape, yb.shape)

(32, 224, 224, 1) (32, 224, 224, 1) (32,)


In [9]:
def get_distances(model, ds):
    d_all, y_all = [], []
    for (x, y) in ds:
        d_sq = model.predict(x, verbose=0).reshape(-1)
        d = np.sqrt(np.maximum(d_sq, 0.0))  # convert squared distance to distance
        d_all.append(d)
        y_all.append(y.numpy().reshape(-1))
    return np.concatenate(d_all), np.concatenate(y_all)

cedar_d, cedar_y = get_distances(siamese_model, cedar_test_ds)

print("CEDAR cross-dataset distances:")
print("Min:", cedar_d.min(), "Mean:", cedar_d.mean(), "Max:", cedar_d.max())

CEDAR cross-dataset distances:
Min: 0.0006654346 Mean: 0.092768945 Max: 0.6079542


In [10]:
def compute_far_frr(d, y, thresholds):
    y = y.astype(int)
    neg = (y == 0)
    pos = (y == 1)
    far, frr = [], []
    for t in thresholds:
        far.append(np.mean(d[neg] <= t))
        frr.append(np.mean(d[pos] >  t))
    return np.array(far), np.array(frr)

thresholds = np.linspace(cedar_d.min(), cedar_d.max(), 800)
far, frr = compute_far_frr(cedar_d, cedar_y, thresholds)

eer_idx = np.argmin(np.abs(far - frr))
eer = (far[eer_idx] + frr[eer_idx]) / 2
best_t = thresholds[eer_idx]

print("Cross-dataset CEDAR EER (BHSig model):", float(eer))
print("Best CEDAR threshold:", float(best_t))
print("FAR@EER:", float(far[eer_idx]), "FRR@EER:", float(frr[eer_idx]))

Cross-dataset CEDAR EER (BHSig model): 0.24020000000000002
Best CEDAR threshold: 0.056149888783693314
FAR@EER: 0.2416 FRR@EER: 0.2388


In [11]:
BHSIG_THRESHOLD = 0.2071010172367096

neg = (cedar_y == 0)
pos = (cedar_y == 1)

cross_far = np.mean(cedar_d[neg] <= BHSIG_THRESHOLD)
cross_frr = np.mean(cedar_d[pos] >  BHSIG_THRESHOLD)
cross_avg = (cross_far + cross_frr) / 2

print("Using BHSig threshold on CEDAR:")
print("Cross FAR:", float(cross_far))
print("Cross FRR:", float(cross_frr))
print("Cross avg error:", float(cross_avg))

Using BHSig threshold on CEDAR:
Cross FAR: 0.7639
Cross FRR: 0.0062
Cross avg error: 0.38505


GPDS cross validation

In [12]:


GPDS_ROOT = r"W:\SRH study\Case Study 2\Offline Signature Verification\Datasets\GPDS150"

def parse_gpds_filename(fname):
    m = re.search(r"cf?-(\d+)-(\d+)", fname.replace(" ", ""), re.IGNORECASE)
    if m:
        return int(m.group(1)), int(m.group(2))
    return None, None

def build_gpds150_df(gpds_root):
    rows = []
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

    # TRAIN (flat folders)
    train_genuine = os.path.join(gpds_root, "train", "genuine")
    train_forge   = os.path.join(gpds_root, "train", "forge")

    for label, folder in [("genuine", train_genuine), ("forgery", train_forge)]:
        for fname in os.listdir(folder):
            if Path(fname).suffix.lower() not in exts:
                continue
            wid, sid = parse_gpds_filename(fname)
            if wid is None:
                continue
            rows.append({
                "dataset": "GPDS150",
                "split": "train",
                "writer_id": wid,
                "sample_id": sid,
                "label": label,
                "path": os.path.join(folder, fname),
                "filename": fname
            })

    # TEST (per-writer folders)
    test_root = os.path.join(gpds_root, "test")
    for writer in sorted(os.listdir(test_root)):
        writer_dir = os.path.join(test_root, writer)
        if not os.path.isdir(writer_dir):
            continue

        for subfolder, label in [("genuine", "genuine"), ("forge", "forgery")]:
            subdir = os.path.join(writer_dir, subfolder)
            if not os.path.exists(subdir):
                continue

            for fname in os.listdir(subdir):
                if Path(fname).suffix.lower() not in exts:
                    continue
                wid, sid = parse_gpds_filename(fname)
                if wid is None:
                    wid = int(writer)
                rows.append({
                    "dataset": "GPDS150",
                    "split": "test",
                    "writer_id": int(wid),
                    "sample_id": sid,
                    "label": label,
                    "path": os.path.join(subdir, fname),
                    "filename": fname
                })

    return pd.DataFrame(rows)

try:
    valid_gpds
    print("Using existing valid_gpds dataframe.")
except NameError:
    valid_gpds = build_gpds150_df(GPDS_ROOT)
    print("Rebuilt valid_gpds dataframe.")

gpds_test_df = valid_gpds[valid_gpds["split"] == "test"].copy()

print("GPDS test images:", len(gpds_test_df))
print(gpds_test_df["label"].value_counts())
print("GPDS test writers:", gpds_test_df["writer_id"].nunique())
display(gpds_test_df.head())

Rebuilt valid_gpds dataframe.
GPDS test images: 3300
label
forgery    2100
genuine    1200
Name: count, dtype: int64
GPDS test writers: 150


,dataset,split,writer_id,sample_id,label,path,filename
4800,GPDS150,test,1,17,genuine,W:\SRH study\Case Study 2\Offline Signature Ve...,c-001-17 (Copy).jpg
4801,GPDS150,test,1,18,genuine,W:\SRH study\Case Study 2\Offline Signature Ve...,c-001-18 (Copy).jpg
4802,GPDS150,test,1,19,genuine,W:\SRH study\Case Study 2\Offline Signature Ve...,c-001-19 (Copy).jpg
4803,GPDS150,test,1,20,genuine,W:\SRH study\Case Study 2\Offline Signature Ve...,c-001-20 (Copy).jpg
4804,GPDS150,test,1,21,genuine,W:\SRH study\Case Study 2\Offline Signature Ve...,c-001-21 (Copy).jpg


In [13]:
def build_pools(df_subset):
    genuine_by_writer = {}
    forgery_by_writer = {}

    for wid, g in df_subset.groupby("writer_id"):
        gg = g[g["label"] == "genuine"]["path"].tolist()
        ff = g[g["label"] == "forgery"]["path"].tolist()

        if len(gg) >= 2:
            genuine_by_writer[wid] = gg
        if len(ff) >= 1:
            forgery_by_writer[wid] = ff

    return genuine_by_writer, forgery_by_writer

def generate_pairs(df, writer_set, n_pairs=20000, seed=42, neg_mix=0.9):
    df_subset = df[df["writer_id"].isin(writer_set)].copy()
    genuine_by_writer, forgery_by_writer = build_pools(df_subset)

    writers = sorted(genuine_by_writer.keys())
    writers_with_forg = sorted(set(genuine_by_writer) & set(forgery_by_writer))

    rng = np.random.default_rng(seed)
    n_pos = n_pairs // 2
    n_neg = n_pairs - n_pos
    n_neg_same  = int(round(n_neg * neg_mix))
    n_neg_cross = n_neg - n_neg_same

    pairs = []

    # Positive pairs: genuine-genuine same writer
    for _ in range(n_pos):
        w = rng.choice(writers)
        g = genuine_by_writer[w]
        i, j = rng.choice(len(g), size=2, replace=False)
        pairs.append({"path_a": g[i], "path_b": g[j], "label": 1})

    # Negative pairs: genuine-forgery same writer
    for _ in range(n_neg_same):
        w = rng.choice(writers_with_forg)
        g = genuine_by_writer[w]
        f = forgery_by_writer[w]
        i = rng.integers(0, len(g))
        j = rng.integers(0, len(f))
        pairs.append({"path_a": g[i], "path_b": f[j], "label": 0})

    # Negative pairs: genuine-genuine cross writer
    for _ in range(n_neg_cross):
        w1, w2 = rng.choice(writers, size=2, replace=False)
        g1 = genuine_by_writer[w1]
        g2 = genuine_by_writer[w2]
        i = rng.integers(0, len(g1))
        j = rng.integers(0, len(g2))
        pairs.append({"path_a": g1[i], "path_b": g2[j], "label": 0})

    return pd.DataFrame(pairs).sample(frac=1.0, random_state=seed).reset_index(drop=True)

gpds_test_writers = set(gpds_test_df["writer_id"].unique())
gpds_cross_pairs = generate_pairs(gpds_test_df, gpds_test_writers, n_pairs=20000, seed=42, neg_mix=0.9)

print("GPDS cross-dataset test pairs:", len(gpds_cross_pairs))
print(gpds_cross_pairs["label"].value_counts(normalize=True))
display(gpds_cross_pairs.head())

GPDS cross-dataset test pairs: 20000
label
0    0.5
1    0.5
Name: proportion, dtype: float64


,path_a,path_b,label
0,W:\SRH study\Case Study 2\Offline Signature Ve...,W:\SRH study\Case Study 2\Offline Signature Ve...,0
1,W:\SRH study\Case Study 2\Offline Signature Ve...,W:\SRH study\Case Study 2\Offline Signature Ve...,1
2,W:\SRH study\Case Study 2\Offline Signature Ve...,W:\SRH study\Case Study 2\Offline Signature Ve...,1
3,W:\SRH study\Case Study 2\Offline Signature Ve...,W:\SRH study\Case Study 2\Offline Signature Ve...,1
4,W:\SRH study\Case Study 2\Offline Signature Ve...,W:\SRH study\Case Study 2\Offline Signature Ve...,0


In [14]:
import cv2
import tensorflow as tf

IMG_SIZE = (224, 224)

def load_preprocess(path):
    path = path.numpy().decode("utf-8")
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Failed to read: {path}")

    img = cv2.resize(img, IMG_SIZE, interpolation=cv2.INTER_AREA)
    img = img.astype(np.float32) / 255.0
    img = np.expand_dims(img, axis=-1)
    return img

def tf_load_preprocess(path):
    img = tf.py_function(load_preprocess, [path], Tout=tf.float32)
    img.set_shape([IMG_SIZE[0], IMG_SIZE[1], 1])
    return img

def make_pair_dataset(pairs_df, batch_size=32):
    a = pairs_df["path_a"].astype(str).values
    b = pairs_df["path_b"].astype(str).values
    y = pairs_df["label"].astype(np.float32).values

    ds = tf.data.Dataset.from_tensor_slices((a, b, y))

    def map_fn(pa, pb, lab):
        ia = tf_load_preprocess(pa)
        ib = tf_load_preprocess(pb)
        return (ia, ib), lab

    ds = ds.map(map_fn, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

gpds_cross_ds = make_pair_dataset(gpds_cross_pairs, batch_size=32)

(xb, yb) = next(iter(gpds_cross_ds))
print(xb[0].shape, xb[1].shape, yb.shape)

(32, 224, 224, 1) (32, 224, 224, 1) (32,)


In [15]:
def get_distances(model, ds):
    d_all, y_all = [], []
    for (x, y) in ds:
        d_sq = model.predict(x, verbose=0).reshape(-1)
        d = np.sqrt(np.maximum(d_sq, 0.0))   # convert squared distance -> distance
        d_all.append(d)
        y_all.append(y.numpy().reshape(-1))
    return np.concatenate(d_all), np.concatenate(y_all)

gpds_d, gpds_y = get_distances(siamese_model, gpds_cross_ds)

print("GPDS cross-dataset distances:")
print("Min:", gpds_d.min(), "Mean:", gpds_d.mean(), "Max:", gpds_d.max())

GPDS cross-dataset distances:
Min: 0.0011800803 Mean: 0.5249568 Max: 1.2324803


In [16]:
def compute_far_frr(d, y, thresholds):
    y = y.astype(int)
    neg = (y == 0)
    pos = (y == 1)
    far, frr = [], []
    for t in thresholds:
        far.append(np.mean(d[neg] <= t))
        frr.append(np.mean(d[pos] >  t))
    return np.array(far), np.array(frr)

thresholds = np.linspace(gpds_d.min(), gpds_d.max(), 800)
far, frr = compute_far_frr(gpds_d, gpds_y, thresholds)

eer_idx = np.argmin(np.abs(far - frr))
eer = (far[eer_idx] + frr[eer_idx]) / 2
best_t = thresholds[eer_idx]

print("Cross-dataset GPDS EER (BHSig model):", float(eer))
print("Best GPDS threshold:", float(best_t))
print("FAR@EER:", float(far[eer_idx]), "FRR@EER:", float(frr[eer_idx]))

Cross-dataset GPDS EER (BHSig model): 0.48065
Best GPDS threshold: 0.5482534170150757
FAR@EER: 0.4807 FRR@EER: 0.4806


In [17]:
BHSIG_THRESHOLD = 0.2071010172367096

neg = (gpds_y == 0)
pos = (gpds_y == 1)

cross_far = np.mean(gpds_d[neg] <= BHSIG_THRESHOLD)
cross_frr = np.mean(gpds_d[pos] >  BHSIG_THRESHOLD)
cross_avg = (cross_far + cross_frr) / 2

print("Using BHSig threshold on GPDS:")
print("Cross FAR:", float(cross_far))
print("Cross FRR:", float(cross_frr))
print("Cross avg error:", float(cross_avg))

Using BHSig threshold on GPDS:
Cross FAR: 0.1992
Cross FRR: 0.7624
Cross avg error: 0.4808
